# SINIS Calibration, Statistic and Temperature Sweep Measurements

## Description

This is one of four template notebooks that together replace the original
monolithic workflow. **This notebook covers SINIS calibration, statistics and temperature sweep experiments** for a single qubit using the LabOne Q workflow architecture:

1. [Statistics](#statistics)
2. [Nonlocal Heating with Heater at A vs. Biased at a Lower RO Amp](#nonlocal-heating-with-heater-at-a-vs-biased-at-a-lower-ro-amp)
3. [Nonlocal Heating with Heater at A vs. Biased at a Lower RO Amp up to Higher V Bias](#nonlocal-heating-with-heater-at-a-vs-biased-at-a-lower-ro-amp-up-to-higher-v-bias)
4. [Short -- Local Heating with Heater at A. Split geometry](#short----local-heating-with-heater-at-a-split-geometry)
5. [Long -- Local Heating with Heater at A. Split geometry](#long----local-heating-with-heater-at-a-split-geometry)
6. [Fine around the gape -- Local Heating with Heater at A. Split geometry](#fine-around-the-gap----local-heating-with-heater-at-a-split-geometry)
7. [Long -- Local Heating with Heater at A With SSRO](#long----local-heating-with-heater-at-a-with-ssro)
8. [Fine sweep around the gap -- Local Heating with Heater at A With SSRO](#fine-sweep-around-the-gap----local-heating-with-heater-at-a-with-ssro)
9. [Nonlocal Heating with Heater at SINIS C and SINIS A is biased at ofst voltage](#nonlocal-heating-with-heater-at-sinis-c-and-sinis-a-is-biased-at-ofst-voltage)

Run this notebook for SINIS calibration and sweeping temperature. This is the main script of the project. The other three
template notebooks under `qubit_thermometry/templates/` load that saved QPU
and continue from there:

- Tune-up experiments.
- Population and temperature measurements (single-shot 0/1, population sweeps).
- Fast Flux Drive.


## Changelog

All changes must be logged here! Do not make changes to this notebook other than expanding or modifying the standard experiments in a meaningful! All device-specific parameters and measurements have to be done in a different notebook!

Stick to semantic versioning:
- **Last Digit**: Fix a bug that does not add any new features and is backwards compatible -> increment last digit by one (e.g. 0.0.4 to 0.0.5)

- **Middle Digit**: Add a new feature while keeping backwards compatibility (older versions can still be executed without any modifications) -> increment middle digit by one and set last digit to 0 (e.g. 1.2.12 to 1.3.0)

- **First Digit**: Make changes that will break older versions, making them incompatible with the new version -> increment first digit by one and set all other digits to 0 (e.g. 3.2.9 to 4.0.0)

**Current Version:** <font color="#008000">**v2.0.0**</font>

**2026-09-27**, *Kha*: Updating from version v1.7.3, separated specified script for different experiments. On-going SINIS experiments. (v2.0.0)

## Setup

In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path("/Users/khatran/Desktop/qubit-thermometry_v2.0")
sys.path.insert(0, str(REPO_ROOT / "qubit_thermometry" / "src"))
# Extra: needed for `import lib.devices...` and `from blueftc.BlueforsController import ...`
# (see `helper/sinis_devices.py`).
sys.path.insert(0, str(REPO_ROOT))


In [ ]:
import time
from copy import deepcopy
from datetime import datetime
from functools import partial

import matplotlib.pyplot as plt
import numpy as np

from laboneq_applications.experiments import iq_blobs, lifetime_measurement, ramsey

from qubit_thermometry.helper import qpu_io, setup, sinis_devices
from qubit_thermometry.helper.temperature import get_qubit_temperature_parameters
from qubit_thermometry.workflow.experiments.population import population_experiment_workflow
from qubit_thermometry.workflow.experiments.sinis_heating_sweep import (
    run_heating_sweep,
    run_qubit_health_statistics,
)
from qubit_thermometry.workflow.analysis.sinis_heating import (
    plot_heating_sweep_results,
    plot_qubit_health_statistics,
    save_heating_sweep_data,
)


In [ ]:
sample_name = input('Sample name: ')
qubit_name = input('Qubit name: ')
cooldown_start_date = input('Cooldown start date (YYYY-MM-DD): ')


In [ ]:
# `DeviceSetup` (physical instrument connections/ports/IPs) is a separate concern
# from the QPU (qubit calibration parameters) and is never saved/loaded -- it is
# cheap and deterministic to rebuild, so every non-tune-up notebook reconstructs it
# with the same descriptor arguments used by the tune-up notebook for this setup.
shfqc_device_id = "DEV12192"

device_setup, _default_qubits = setup.build_device_setup(
    shfqc_device_id=shfqc_device_id,
    number_of_qubits=1,
    ip_address="localhost",
)
session = setup.connect_session(device_setup)


In [ ]:
data_root_directory, qpu_file_path = setup.init_data_saving(
    sample_name, cooldown_start_date, qubit_name
)
get_path_to_file = partial(setup.get_path_to_file, directory=data_root_directory)

qpu = qpu_io.load_qpu(qpu_file_path)
qubits = qpu.quantum_elements
qops = qpu.quantum_operations

folder_store, logging_store = setup.setup_logbook(data_root_directory)


In [ ]:
def build_experiment_options(n_avg_exponent_pop, n_avg_exponent_t1, n_avg_exponent_ramsey):
    """pop/T1/Ramsey `.options()`, configured the same way in every SINIS subsection below."""
    pop_options = population_experiment_workflow.options()
    pop_options.count(2**n_avg_exponent_pop)

    t1_options = lifetime_measurement.experiment_workflow.options()
    t1_options.count(2**n_avg_exponent_t1)
    t1_options.transition('ge')
    t1_options.cal_states('ge')
    t1_options.update(False)
    t1_options.close_figures(True)

    ramsey_options = ramsey.experiment_workflow.options()
    ramsey_options.count(2**n_avg_exponent_ramsey)
    ramsey_options.transition('ge')
    ramsey_options.cal_states('ge')
    ramsey_options.update(False)
    ramsey_options.close_figures(True)

    return pop_options, t1_options, ramsey_options


def build_ssro_options(n_avg_exponent_ssro):
    """`iq_blobs.experiment_workflow.options()` used by every "... With SSRO" subsection."""
    ssro_options = iq_blobs.experiment_workflow.options()
    ssro_options.count(2**n_avg_exponent_ssro)
    ssro_options.close_figures(True)
    ssro_options.do_analysis(False)
    return ssro_options


## Setup Instruments for DC-Measurements

#### Temperature Controller

In [ ]:
bft_controller = sinis_devices.connect_temperature_controller()
print('MXC temperature:', bft_controller.get_mxc_temperature() * 1e3, 'mK')


#### DC Voltmeter and SIM Voltage Source

In [ ]:
dmm_address = 'TCPIP0::192.168.1.44::hislip0::INSTR'
dmm = sinis_devices.connect_dc_voltmeter(dmm_address)
print('DMM scan:', dmm.scan())


In [ ]:
sim_frame_address = 'GPIB1::3::INSTR'
frame, sim, sim2 = sinis_devices.connect_sim_voltage_source(sim_frame_address)


In [ ]:
# A12-A11 measurement pair (SINIS A, `sim`). A9 non-symmetric heating (SINIS C, `sim2`).
I_probe = 0  # current source, 50 pA
V_div = 1000  # voltage divider between the SIM output and the applied heater voltage

print('Probe current:  ', I_probe)
print('Voltage divider:', V_div)


## Statistics

#### Experiment Parameters

In [ ]:
n_measurements = 25

n_avg_exponent_pop = 15
n_avg_exponent_t1 = 13
n_avg_exponent_ramsey = 13

qubit_to_measure = qubits[0]

ramsey_detuning = 2e6

t1_delays = setup.log_sweep_help(0.0, 80e-6, 51)
ramsey_delays = np.linspace(0.0, 70e-6, 101)

f_q, anharm = get_qubit_temperature_parameters(qubit_to_measure)
print('f_q:                    ', f_q, 'GHz')
print('anharm:                 ', anharm, 'GHz')
print('Number of measurements: ', n_measurements)


#### Run Workflow

In [ ]:
pop_options, t1_options, ramsey_options = build_experiment_options(
    n_avg_exponent_pop, n_avg_exponent_t1, n_avg_exponent_ramsey
)

statistics_result = run_qubit_health_statistics(
    session=session,
    qpu=qpu,
    qubit=qubit_to_measure,
    n_measurements=n_measurements,
    bft_controller=bft_controller,
    pop_options=pop_options,
    t1_options=t1_options,
    ramsey_options=ramsey_options,
    t1_delays=t1_delays,
    ramsey_delays=ramsey_delays,
    ramsey_detuning=ramsey_detuning,
    folder_store=folder_store,
    logging_store=logging_store,
)


#### Analysis Plots

In [ ]:
statistics_plots = plot_qubit_health_statistics(
    statistics_result, qubit_to_measure, f_q, anharm, get_path_to_file
)


#### Update Parameters

In [ ]:
save_heating_sweep_data(
    get_path_to_file('Statistic_', '.mat'),
    qubit_to_measure,
    sample_name,
    qubit_name,
    cooldown_start_date,
    statistics_result,
    tabc=statistics_plots['TABC'],
    f_q=f_q,
    anharm=anharm,
    comment='Qubit health statistics, no bias sweep',
)


## Nonlocal Heating with Heater at A vs. Biased at a Lower RO Amp

#### Experiment Parameters

In [ ]:
n_avg_exponent_pop = 15
n_avg_exponent_t1 = 13
n_avg_exponent_ramsey = 13

qubit_to_measure = qubits[0]

ramsey_detuning = 2e6

t1_delays = setup.log_sweep_help(0.0, 80e-6, 51)
ramsey_delays = np.linspace(0.0, 70e-6, 101)

V_off = -1.378
b_sh_arr = np.round(np.linspace(-1.9, 1.9, 191), 4)
bias_array = V_off + b_sh_arr

wait_time = 5

f_q, anharm = get_qubit_temperature_parameters(qubit_to_measure)
print('f_q:                ', f_q, 'GHz')
print('anharm:             ', anharm, 'GHz')
print('Heater bias sweep:  ', bias_array[0], '...', bias_array[-1], 'V')
print('Number of points:   ', len(bias_array))


#### Run Workflow

In [ ]:
pop_options, t1_options, ramsey_options = build_experiment_options(
    n_avg_exponent_pop, n_avg_exponent_t1, n_avg_exponent_ramsey
)

heating_result_1 = run_heating_sweep(
    session=session,
    qpu=qpu,
    qubit=qubit_to_measure,
    bias_array=bias_array,
    sim=sim,
    dmm=dmm,
    bft_controller=bft_controller,
    v_off=V_off,
    v_div=V_div,
    pop_options=pop_options,
    t1_options=t1_options,
    ramsey_options=ramsey_options,
    t1_delays=t1_delays,
    ramsey_delays=ramsey_delays,
    ramsey_detuning=ramsey_detuning,
    folder_store=folder_store,
    logging_store=logging_store,
    wait_time=wait_time,
    n_ramp_steps=20,
    read_mxc_temperature=True,
)


#### Analysis Plots

In [ ]:
heating_plots_1 = plot_heating_sweep_results(
    heating_result_1,
    qubit_to_measure,
    f_q,
    anharm,
    V_off,
    get_path_to_file,
    x_label='V_Heater, corrected (mV)',
    x_scale=1.0,
    teff_ylim=(50, 600),
    t1_ylim=None,
    file_prefix='01_nonlocal_A_lowV_',
)


#### Update Parameters

In [ ]:
save_heating_sweep_data(
    get_path_to_file(
        '01_nonlocal_A_lowV_' + 't1_pop_ramsey_', '.mat'
    ),
    qubit_to_measure,
    sample_name,
    qubit_name,
    cooldown_start_date,
    heating_result_1,
    tabc=heating_plots_1['TABC'],
    f_q=f_q,
    anharm=anharm,
    comment='Nonlocal heating sweep at low bias, SINIS A',
)


## Nonlocal Heating with Heater at A vs. Biased at a Lower RO Amp up to Higher V Bias

#### Experiment Parameters

In [ ]:
n_avg_exponent_pop = 15
n_avg_exponent_t1 = 13
n_avg_exponent_ramsey = 13

qubit_to_measure = qubits[0]

ramsey_detuning = 2e6

t1_delays = setup.log_sweep_help(0.0, 80e-6, 51)
ramsey_delays = np.linspace(0.0, 70e-6, 101)

V_off = -1.378
b_sh_arr = np.round(np.linspace(0.0, 5.0, 51), 4)
bias_array = V_off + b_sh_arr

wait_time = 5

f_q, anharm = get_qubit_temperature_parameters(qubit_to_measure)
print('f_q:                ', f_q, 'GHz')
print('anharm:             ', anharm, 'GHz')
print('Heater bias sweep:  ', bias_array[0], '...', bias_array[-1], 'V')
print('Number of points:   ', len(bias_array))


#### Run Workflow

In [ ]:
pop_options, t1_options, ramsey_options = build_experiment_options(
    n_avg_exponent_pop, n_avg_exponent_t1, n_avg_exponent_ramsey
)

heating_result_2 = run_heating_sweep(
    session=session,
    qpu=qpu,
    qubit=qubit_to_measure,
    bias_array=bias_array,
    sim=sim,
    dmm=dmm,
    bft_controller=bft_controller,
    v_off=V_off,
    v_div=V_div,
    pop_options=pop_options,
    t1_options=t1_options,
    ramsey_options=ramsey_options,
    t1_delays=t1_delays,
    ramsey_delays=ramsey_delays,
    ramsey_detuning=ramsey_detuning,
    folder_store=folder_store,
    logging_store=logging_store,
    wait_time=wait_time,
    n_ramp_steps=20,
    read_mxc_temperature=True,
)


#### Analysis Plots

In [ ]:
heating_plots_2 = plot_heating_sweep_results(
    heating_result_2,
    qubit_to_measure,
    f_q,
    anharm,
    V_off,
    get_path_to_file,
    x_label='V_Heater, corrected (mV)',
    x_scale=1.0,
    teff_ylim=(50, 300),
    t1_ylim=(0, 40),
    file_prefix='02_nonlocal_A_highV_',
)


#### Update Parameters

In [ ]:
save_heating_sweep_data(
    get_path_to_file(
        '02_nonlocal_A_highV_' + 't1_pop_ramsey_', '.mat'
    ),
    qubit_to_measure,
    sample_name,
    qubit_name,
    cooldown_start_date,
    heating_result_2,
    tabc=heating_plots_2['TABC'],
    f_q=f_q,
    anharm=anharm,
    comment='Nonlocal heating sweep up to higher V bias, SINIS A',
)


## Short -- Local Heating with Heater at A. Split geometry

#### Experiment Parameters

In [ ]:
n_avg_exponent_pop = 15
n_avg_exponent_t1 = 13
n_avg_exponent_ramsey = 13

qubit_to_measure = qubits[0]

ramsey_detuning = 5e6

t1_delays = setup.log_sweep_help(0.0, 10e-6, 51)
ramsey_delays = np.linspace(0.0, 5e-6, 51)

V_off = -1.378
b_sh_arr = np.round(np.linspace(-5, 5, 6), 4)  # SHORT
bias_array = V_off + b_sh_arr

wait_time = 10

f_q, anharm = get_qubit_temperature_parameters(qubit_to_measure)
print('f_q:                ', f_q, 'GHz')
print('anharm:             ', anharm, 'GHz')
print('Heater bias sweep:  ', bias_array[0], '...', bias_array[-1], 'V')
print('Number of points:   ', len(bias_array))


#### Run Workflow

In [ ]:
pop_options, t1_options, ramsey_options = build_experiment_options(
    n_avg_exponent_pop, n_avg_exponent_t1, n_avg_exponent_ramsey
)

heating_result_3 = run_heating_sweep(
    session=session,
    qpu=qpu,
    qubit=qubit_to_measure,
    bias_array=bias_array,
    sim=sim,
    dmm=dmm,
    bft_controller=bft_controller,
    v_off=V_off,
    v_div=V_div,
    pop_options=pop_options,
    t1_options=t1_options,
    ramsey_options=ramsey_options,
    t1_delays=t1_delays,
    ramsey_delays=ramsey_delays,
    ramsey_detuning=ramsey_detuning,
    folder_store=folder_store,
    logging_store=logging_store,
    wait_time=wait_time,
    n_ramp_steps=100,
    read_mxc_temperature=False,
)


#### Analysis Plots

In [ ]:
heating_plots_3 = plot_heating_sweep_results(
    heating_result_3,
    qubit_to_measure,
    f_q,
    anharm,
    V_off,
    get_path_to_file,
    x_label='V_Heater, corrected (mV)',
    x_scale=1.0,
    teff_ylim=(50, 200),
    t1_ylim=None,
    file_prefix='03_local_A_short_',
)


#### Update Parameters

In [ ]:
save_heating_sweep_data(
    get_path_to_file(
        '03_local_A_short_' + 't1_pop_ramsey_', '.mat'
    ),
    qubit_to_measure,
    sample_name,
    qubit_name,
    cooldown_start_date,
    heating_result_3,
    tabc=heating_plots_3['TABC'],
    f_q=f_q,
    anharm=anharm,
    comment='Short local heating sweep, SINIS A, split geometry',
)


## Long -- Local Heating with Heater at A. Split geometry

#### Experiment Parameters

In [ ]:
n_avg_exponent_pop = 16
n_avg_exponent_t1 = 13
n_avg_exponent_ramsey = 13

qubit_to_measure = qubits[0]

ramsey_detuning = 4e6

t1_delays = setup.log_sweep_help(0.0, 10e-6, 51)
ramsey_delays = np.linspace(0.0, 7e-6, 71)

V_off = -1.378
b1 = np.arange(-17, -1 - 0.1, 0.5)
b2 = np.arange(-1, 1 + 0.02, 0.05)
b3 = np.arange(1, 17 + 0.01, 0.5)
b_sh_arr = np.round(np.concatenate([b1, b2, b3]), 4)
bias_array = V_off + b_sh_arr

wait_time = 10

f_q, anharm = get_qubit_temperature_parameters(qubit_to_measure)
print('f_q:                ', f_q, 'GHz')
print('anharm:             ', anharm, 'GHz')
print('Heater bias sweep:  ', bias_array[0], '...', bias_array[-1], 'V')
print('Number of points:   ', len(bias_array))


#### Run Workflow

In [ ]:
pop_options, t1_options, ramsey_options = build_experiment_options(
    n_avg_exponent_pop, n_avg_exponent_t1, n_avg_exponent_ramsey
)

heating_result_4 = run_heating_sweep(
    session=session,
    qpu=qpu,
    qubit=qubit_to_measure,
    bias_array=bias_array,
    sim=sim,
    dmm=dmm,
    bft_controller=bft_controller,
    v_off=V_off,
    v_div=V_div,
    pop_options=pop_options,
    t1_options=t1_options,
    ramsey_options=ramsey_options,
    t1_delays=t1_delays,
    ramsey_delays=ramsey_delays,
    ramsey_detuning=ramsey_detuning,
    folder_store=folder_store,
    logging_store=logging_store,
    wait_time=wait_time,
    n_ramp_steps=100,
    read_mxc_temperature=False,
)


#### Analysis Plots

In [ ]:
heating_plots_4 = plot_heating_sweep_results(
    heating_result_4,
    qubit_to_measure,
    f_q,
    anharm,
    V_off,
    get_path_to_file,
    x_label='V_Heater, corrected (mV)',
    x_scale=1.0,
    teff_ylim=(50, 250),
    t1_ylim=None,
    file_prefix='04_local_A_long_',
)


#### Update Parameters

In [ ]:
save_heating_sweep_data(
    get_path_to_file(
        '04_local_A_long_' + 't1_pop_ramsey_', '.mat'
    ),
    qubit_to_measure,
    sample_name,
    qubit_name,
    cooldown_start_date,
    heating_result_4,
    tabc=heating_plots_4['TABC'],
    f_q=f_q,
    anharm=anharm,
    comment='Long local heating sweep, SINIS A, split geometry',
)


## Fine around the gap -- Local Heating with Heater at A. Split geometry

#### Experiment Parameters

In [ ]:
n_avg_exponent_pop = 17
n_avg_exponent_t1 = 13
n_avg_exponent_ramsey = 13

qubit_to_measure = qubits[0]

ramsey_detuning = 4e6

t1_delays = setup.log_sweep_help(0.0, 10e-6, 51)
ramsey_delays = np.linspace(0.0, 7e-6, 71)

V_off = -1.378
b_sh_arr = np.round(np.arange(-2, 2, 0.02), 4)
bias_array = V_off + b_sh_arr

wait_time = 10

f_q, anharm = get_qubit_temperature_parameters(qubit_to_measure)
print('f_q:                ', f_q, 'GHz')
print('anharm:             ', anharm, 'GHz')
print('Heater bias sweep:  ', bias_array[0], '...', bias_array[-1], 'V')
print('Number of points:   ', len(bias_array))


#### Run Workflow

In [ ]:
pop_options, t1_options, ramsey_options = build_experiment_options(
    n_avg_exponent_pop, n_avg_exponent_t1, n_avg_exponent_ramsey
)

heating_result_5 = run_heating_sweep(
    session=session,
    qpu=qpu,
    qubit=qubit_to_measure,
    bias_array=bias_array,
    sim=sim,
    dmm=dmm,
    bft_controller=bft_controller,
    v_off=V_off,
    v_div=V_div,
    pop_options=pop_options,
    t1_options=t1_options,
    ramsey_options=ramsey_options,
    t1_delays=t1_delays,
    ramsey_delays=ramsey_delays,
    ramsey_detuning=ramsey_detuning,
    folder_store=folder_store,
    logging_store=logging_store,
    wait_time=wait_time,
    n_ramp_steps=100,
    read_mxc_temperature=False,
)


#### Analysis Plots

In [ ]:
heating_plots_5 = plot_heating_sweep_results(
    heating_result_5,
    qubit_to_measure,
    f_q,
    anharm,
    V_off,
    get_path_to_file,
    x_label='V_Heater, corrected (eV/Delta)',
    x_scale=0.216,
    teff_ylim=(50, 90),
    t1_ylim=None,
    file_prefix='05_local_A_fine_',
)


#### Update Parameters

In [ ]:
save_heating_sweep_data(
    get_path_to_file(
        '05_local_A_fine_' + 't1_pop_ramsey_', '.mat'
    ),
    qubit_to_measure,
    sample_name,
    qubit_name,
    cooldown_start_date,
    heating_result_5,
    tabc=heating_plots_5['TABC'],
    f_q=f_q,
    anharm=anharm,
    comment='Fine sweep around the gap, local heating, SINIS A, split geometry',
)


## Long -- Local Heating with Heater at A With SSRO

#### Experiment Parameters

In [ ]:
n_avg_exponent_pop = 16
n_avg_exponent_t1 = 13
n_avg_exponent_ramsey = 13
n_avg_exponent_ssro = 17

qubit_to_measure = qubits[0]

ramsey_detuning = 4e6

t1_delays = setup.log_sweep_help(0.0, 10e-6, 51)
ramsey_delays = np.linspace(0.0, 7e-6, 71)

V_off = -1.378
b_sh_arr = np.round(np.arange(-18, 18, 0.2), 4)
bias_array = V_off + b_sh_arr

wait_time = 10

f_q, anharm = get_qubit_temperature_parameters(qubit_to_measure)
print('f_q:                ', f_q, 'GHz')
print('anharm:             ', anharm, 'GHz')
print('Heater bias sweep:  ', bias_array[0], '...', bias_array[-1], 'V')
print('Number of points:   ', len(bias_array))


#### Run Workflow

In [ ]:
pop_options, t1_options, ramsey_options = build_experiment_options(
    n_avg_exponent_pop, n_avg_exponent_t1, n_avg_exponent_ramsey
)

ssro_options = build_ssro_options(n_avg_exponent_ssro)
shots_file_paths = {
    'g': get_path_to_file('Heating_experiment\\Local_heating_SSRO_g_data', '.mat'),
    'e': get_path_to_file('Heating_experiment\\Local_heating_SSRO_e_data', '.mat'),
    'f': get_path_to_file('Heating_experiment\\Local_heating_SSRO_f_data', '.mat'),
}

heating_result_6 = run_heating_sweep(
    session=session,
    qpu=qpu,
    qubit=qubit_to_measure,
    bias_array=bias_array,
    sim=sim,
    dmm=dmm,
    bft_controller=bft_controller,
    v_off=V_off,
    v_div=V_div,
    pop_options=pop_options,
    t1_options=t1_options,
    ramsey_options=ramsey_options,
    t1_delays=t1_delays,
    ramsey_delays=ramsey_delays,
    ramsey_detuning=ramsey_detuning,
    folder_store=folder_store,
    logging_store=logging_store,
    wait_time=wait_time,
    n_ramp_steps=100,
    read_mxc_temperature=True,
    include_ssro=True,
    ssro_options=ssro_options,
    shots_file_paths=shots_file_paths,
)


#### Analysis Plots

In [ ]:
heating_plots_6 = plot_heating_sweep_results(
    heating_result_6,
    qubit_to_measure,
    f_q,
    anharm,
    V_off,
    get_path_to_file,
    x_label='V_Heater, corrected (eV/Delta)',
    x_scale=0.216,
    teff_ylim=(50, 90),
    t1_ylim=None,
    file_prefix='06_local_A_long_ssro_',
)


#### Update Parameters

In [ ]:
save_heating_sweep_data(
    get_path_to_file(
        '06_local_A_long_ssro_' + 't1_pop_ramsey_', '.mat'
    ),
    qubit_to_measure,
    sample_name,
    qubit_name,
    cooldown_start_date,
    heating_result_6,
    tabc=heating_plots_6['TABC'],
    f_q=f_q,
    anharm=anharm,
    comment='Long local heating sweep with SSRO, SINIS A, split geometry',
)


## Fine sweep around the gap -- Local Heating with Heater at A With SSRO

#### Experiment Parameters

In [ ]:
n_avg_exponent_pop = 16
n_avg_exponent_t1 = 13
n_avg_exponent_ramsey = 13
n_avg_exponent_ssro = 17

qubit_to_measure = qubits[0]

ramsey_detuning = 4e6

t1_delays = setup.log_sweep_help(0.0, 10e-6, 51)
ramsey_delays = np.linspace(0.0, 7e-6, 71)

V_off = -1.378
b_sh_arr = np.round(np.arange(-1.5, 1.5, 0.03), 4)
bias_array = V_off + b_sh_arr

wait_time = 10

f_q, anharm = get_qubit_temperature_parameters(qubit_to_measure)
print('f_q:                ', f_q, 'GHz')
print('anharm:             ', anharm, 'GHz')
print('Heater bias sweep:  ', bias_array[0], '...', bias_array[-1], 'V')
print('Number of points:   ', len(bias_array))


#### Run Workflow

In [ ]:
pop_options, t1_options, ramsey_options = build_experiment_options(
    n_avg_exponent_pop, n_avg_exponent_t1, n_avg_exponent_ramsey
)

ssro_options = build_ssro_options(n_avg_exponent_ssro)
shots_file_paths = {
    'g': get_path_to_file('Heating_experiment\\Local_heating_SSRO_g_data', '.mat'),
    'e': get_path_to_file('Heating_experiment\\Local_heating_SSRO_e_data', '.mat'),
    'f': get_path_to_file('Heating_experiment\\Local_heating_SSRO_f_data', '.mat'),
}

heating_result_7 = run_heating_sweep(
    session=session,
    qpu=qpu,
    qubit=qubit_to_measure,
    bias_array=bias_array,
    sim=sim,
    dmm=dmm,
    bft_controller=bft_controller,
    v_off=V_off,
    v_div=V_div,
    pop_options=pop_options,
    t1_options=t1_options,
    ramsey_options=ramsey_options,
    t1_delays=t1_delays,
    ramsey_delays=ramsey_delays,
    ramsey_detuning=ramsey_detuning,
    folder_store=folder_store,
    logging_store=logging_store,
    wait_time=wait_time,
    n_ramp_steps=100,
    read_mxc_temperature=True,
    include_ssro=True,
    ssro_options=ssro_options,
    shots_file_paths=shots_file_paths,
)


#### Analysis Plots

In [ ]:
heating_plots_7 = plot_heating_sweep_results(
    heating_result_7,
    qubit_to_measure,
    f_q,
    anharm,
    V_off,
    get_path_to_file,
    x_label='V_Heater, corrected (eV/Delta)',
    x_scale=0.216,
    teff_ylim=(55, 85),
    t1_ylim=None,
    file_prefix='07_local_A_fine_ssro_',
)


#### Update Parameters

In [ ]:
save_heating_sweep_data(
    get_path_to_file(
        '07_local_A_fine_ssro_' + 't1_pop_ramsey_', '.mat'
    ),
    qubit_to_measure,
    sample_name,
    qubit_name,
    cooldown_start_date,
    heating_result_7,
    tabc=heating_plots_7['TABC'],
    f_q=f_q,
    anharm=anharm,
    comment='Fine sweep around the gap, local heating with SSRO, SINIS A, split geometry',
)


## Nonlocal Heating with Heater at SINIS C and SINIS A is biased at ofst voltage

#### Experiment Parameters

In [ ]:
n_avg_exponent_pop = 16
n_avg_exponent_t1 = 13
n_avg_exponent_ramsey = 13
n_avg_exponent_ssro = 17

qubit_to_measure = qubits[0]

ramsey_detuning = 4e6

t1_delays = setup.log_sweep_help(0.0, 10e-6, 51)
ramsey_delays = np.linspace(0.0, 5e-6, 101)

V_off = -1.378
b_sh_arr = np.round(np.arange(-3, 3, 0.06), 4)
bias_array = V_off + b_sh_arr

wait_time = 10

f_q, anharm = get_qubit_temperature_parameters(qubit_to_measure)
print('f_q:                ', f_q, 'GHz')
print('anharm:             ', anharm, 'GHz')
print('Heater bias sweep:  ', bias_array[0], '...', bias_array[-1], 'V')
print('Number of points:   ', len(bias_array))


#### Run Workflow

In [ ]:
pop_options, t1_options, ramsey_options = build_experiment_options(
    n_avg_exponent_pop, n_avg_exponent_t1, n_avg_exponent_ramsey
)

ssro_options = build_ssro_options(n_avg_exponent_ssro)
shots_file_paths = {
    'g': get_path_to_file('Heating_experiment\\Local_heating_SSRO_g_data', '.mat'),
    'e': get_path_to_file('Heating_experiment\\Local_heating_SSRO_e_data', '.mat'),
    'f': get_path_to_file('Heating_experiment\\Local_heating_SSRO_f_data', '.mat'),
}

sinis_devices.ramp_voltage_with_retry(sim, V_off, n_steps=100, wait_time=wait_time)

heating_result_8 = run_heating_sweep(
    session=session,
    qpu=qpu,
    qubit=qubit_to_measure,
    bias_array=bias_array,
    sim=sim2,
    dmm=dmm,
    bft_controller=bft_controller,
    v_off=V_off,
    v_div=V_div,
    pop_options=pop_options,
    t1_options=t1_options,
    ramsey_options=ramsey_options,
    t1_delays=t1_delays,
    ramsey_delays=ramsey_delays,
    ramsey_detuning=ramsey_detuning,
    folder_store=folder_store,
    logging_store=logging_store,
    wait_time=wait_time,
    n_ramp_steps=100,
    read_mxc_temperature=True,
    include_ssro=True,
    ssro_options=ssro_options,
    shots_file_paths=shots_file_paths,
)


#### Analysis Plots

In [ ]:
heating_plots_8 = plot_heating_sweep_results(
    heating_result_8,
    qubit_to_measure,
    f_q,
    anharm,
    V_off,
    get_path_to_file,
    x_label='V_Heater, corrected (eV/Delta)',
    x_scale=0.216,
    teff_ylim=(55, 600),
    t1_ylim=None,
    file_prefix='08_nonlocal_C_',
)


#### Update Parameters

In [ ]:
save_heating_sweep_data(
    get_path_to_file(
        '08_nonlocal_C_' + 't1_pop_ramsey_', '.mat'
    ),
    qubit_to_measure,
    sample_name,
    qubit_name,
    cooldown_start_date,
    heating_result_8,
    tabc=heating_plots_8['TABC'],
    f_q=f_q,
    anharm=anharm,
    comment='Nonlocal heating sweep, heater at SINIS C (SIM2), SINIS A held at off-state bias',
)
